# UC-DIM-1 — Dekadal Calendar Dimensions

**Persona:** Platform Admin / API Integrator

**Goal:** List all dimensions on the platform, introspect the `dekadal` temporal
dimension encoding, navigate all 36 dekadal periods of 2024, filter to a specific
month, and validate the February edge case.

**Use case:** UC-DIM-1 — FAO ASIS dekadal drought monitoring

**Prerequisites:**
- `extension_dimensions` installed and registered on the platform
- `ogc-dimensions` package available

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN", "")
DIM_ID = "dekadal"       # bounded 2024 showcase dimension
DIM_BASE = "/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0, follow_redirects=True)
print(f"BASE_URL: {BASE_URL}")
print(f"DIM_ID  : {DIM_ID}")

## Step 1 — List all dimensions

GET the global dimensions index. Each entry advertises an `id`, `title`, and `provider` metadata.

In [ ]:
resp = client.get(DIM_BASE)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

dims_payload = resp.json()
collections = dims_payload.get("collections", [])

print(f"Found {len(collections)} dimension(s):")
for d in collections:
    prov = d.get("provider", {})
    print(f"  id={d.get('id')}  title={d.get('title')}  invertible={prov.get('invertible')}")

dim_meta = next((d for d in collections if d["id"] == DIM_ID), None)
assert dim_meta is not None, f"Dimension '{DIM_ID}' not found in /dimensions listing"
print(f"\nFound dimension: {DIM_ID}")

## Step 2 — Introspect the temporal dimension

Inspect the provider configuration. The `daily-period` provider with `period_days=10` confirms dekadal encoding.

In [ ]:
provider = dim_meta.get("provider", {})
period_days = provider.get("config", {}).get("period_days")
prov_type = provider.get("type", "")

print(f"id          : {dim_meta.get('id')}")
print(f"title       : {dim_meta.get('title')}")
print(f"provider    : {prov_type}")
print(f"period_days : {period_days}")
print(f"invertible  : {provider.get('invertible')}")

assert prov_type == "daily-period", f"Expected daily-period provider, got {prov_type!r}"
assert period_days == 10, f"Expected 10-day periods for dekadal, got {period_days}"
print("\nDekadal temporal dimension confirmed.")

## Step 3 — Navigate full year (2024) dekadal periods

Use the `search` endpoint with a range of dekad codes to retrieve all 36 periods for 2024.

In [ ]:
resp = client.get(f"{DIM_BASE}/{DIM_ID}/search", params={"min": "2024-K01", "max": "2024-K36"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
periods = data.get("features", [])

print(f"Periods returned: {len(periods)}")
for p in periods[:5]:
    props = p["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {p['id']}  start={interval[0]}  end={interval[1]}")
if len(periods) > 5:
    print(f"  ... ({len(periods) - 5} more)")

assert len(periods) == 36, f"Expected 36 dekadal periods for 2024, got {len(periods)}"
print("\nAssertion passed: 36 dekadal periods for 2024.")

## Step 4 — Filter to January 2024

Narrow the search to dekads K01–K03 (January). A dekadal month always contains exactly 3 periods.

In [ ]:
resp = client.get(f"{DIM_BASE}/{DIM_ID}/search", params={"min": "2024-K01", "max": "2024-K03"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
jan_periods = data.get("features", [])

print(f"January 2024 periods: {len(jan_periods)}")
for p in jan_periods:
    props = p["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {p['id']}  {interval[0]} — {interval[1]}")

assert len(jan_periods) == 3, f"Expected 3 dekadal periods for January 2024, got {len(jan_periods)}"
print("\nAssertion passed: 3 dekadal periods for January 2024.")

## Edge cases

### February — variable-length D3

2024 is a leap year. February D3 (K06) covers Feb 21–29 (9 days). The API must return exactly 3 periods.

In [ ]:
resp = client.get(f"{DIM_BASE}/{DIM_ID}/search", params={"min": "2024-K04", "max": "2024-K06"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
feb_periods = data.get("features", [])

print(f"February 2024 periods: {len(feb_periods)}")
for p in feb_periods:
    props = p["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {p['id']}  {interval[0]} — {interval[1]}")

assert len(feb_periods) == 3, f"Expected 3 dekadal periods for February 2024, got {len(feb_periods)}"

# Verify D3 (K06) covers leap-year Feb end
k06 = next((p for p in feb_periods if p['id'] == '2024-K06'), None)
if k06:
    end = k06['properties'].get('time', {}).get('interval', [None, None])[1]
    assert end == '2024-02-29', f"Expected Feb D3 to end on 2024-02-29 (leap year), got {end!r}"
    print(f"\nLeap-year edge case confirmed: K06 ends on {end}")

In [ ]:
# Out-of-range year — the dimension is bounded to 2024; codes for 2025 must not exist
resp = client.get(f"{DIM_BASE}/{DIM_ID}/search", params={"min": "2025-K01", "max": "2025-K36"})
print(resp.status_code)
assert resp.status_code == 200, (
    f"Expected 200 (empty result) for out-of-range year 2025, got {resp.status_code}: {resp.text}"
)

data = resp.json()
future_periods = data.get("features", [])
assert len(future_periods) == 0, (
    f"Expected 0 periods for out-of-range year 2025, got {len(future_periods)}"
)
print("Empty result (not 404) confirmed for out-of-range year 2025 — correct behaviour.")